In [ ]:
# scripts/3_dataset_unet.py
import os
import torch
from torch.utils.data import Dataset
import rasterio
import numpy as np
from torchvision import transforms
from rasterio.windows import Window

class GeoPatchDataset(Dataset):
    """
    Reads aligned processed rasters and returns patches for training U-Net.
    Expects these files per tile (naming conventions):
      {tile}_DSM.tif
      {tile}_impervious.tif
      {tile}_canopy.tif
      {tile}_NDVI.tif
      {tile}_slope.tif
      {tile}_lidar_dtm.tif
      {tile}_landcover.tif (optional)
    Output:
      x tensor: C x H x W with channels ordered [DSM, impervious, canopy, NDVI, slope, landcover(one-hot?)]
      y1 tensor: height (DSM - DTM) scalar map
      y2 tensor: mask multi-class or binary built/veg mask (constructed from landcover/imperm if available)
    """
    def __init__(self, processed_dir, tile_list, patch_size=256, transforms=None):
        self.dir = processed_dir
        self.tiles = tile_list
        self.patch_size = patch_size
        self.transforms = transforms
        self.entries = []  # list of (tile, x_off, y_off)
        # Build index of non-empty patches
        for t in self.tiles:
            # opens DSM to get dimensions
            p_dsm = os.path.join(self.dir, f"{t}_DSM.tif")
            if not os.path.exists(p_dsm):
                continue
            with rasterio.open(p_dsm) as r:
                H, W = r.height, r.width
            stride = patch_size  # no overlap by default; could do overlap
            for y in range(0, H, stride):
                for x in range(0, W, stride):
                    self.entries.append((t, x, y))

    def __len__(self):
        return len(self.entries)

    def read_window(self, path, x, y):
        if not os.path.exists(path):
            return None
        with rasterio.open(path) as r:
            w = self.patch_size
            win = Window(col_off=x, row_off=y, width=w, height=w)
            arr = r.read(1, window=win, boundless=True, fill_value=np.nan)
            return arr

    def __getitem__(self, idx):
        tile, x, y = self.entries[idx]
        pbase = os.path.join(self.dir, f"{tile}")
        # assemble channels
        dsm = self.read_window(os.path.join(self.dir, f"{tile}_DSM.tif"), x, y)
        imp = self.read_window(os.path.join(self.dir, f"{tile}_impervious.tif"), x, y)
        canopy = self.read_window(os.path.join(self.dir, f"{tile}_canopy.tif"), x, y)
        ndvi = self.read_window(os.path.join(self.dir, f"{tile}_NDVI.tif"), x, y)
        slope = self.read_window(os.path.join(self.dir, f"{tile}_slope.tif"), x, y)
        dtm = self.read_window(os.path.join(self.dir, f"{tile}_lidar_dtm.tif"), x, y)
        lc = self.read_window(os.path.join(self.dir, f"{tile}_landcover.tif"), x, y)  # optional

        # fallback for missing arrays: fill with zeros
        def fill(a):
            if a is None: return np.zeros((self.patch_size, self.patch_size), dtype='float32')
            a = np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0)
            return a.astype('float32')
        dsm, imp, canopy, ndvi, slope, dtm = map(fill, [dsm, imp, canopy, ndvi, slope, dtm])

        # target: height = DSM - DTM
        height = dsm - dtm

        # mask built/veg: if landcover exists try heuristics
        mask_built = np.zeros_like(dsm, dtype='float32')
        mask_veg = np.zeros_like(dsm, dtype='float32')
        if lc is not None:
            # example: assume landcover codes: built = 50-60; veg = 30-40 (adapt to your legend)
            mask_built[(lc >= 50) & (lc <= 80)] = 1.0
            mask_veg[(lc >= 30) & (lc <= 49)] = 1.0
        else:
            # fallback: threshold imperviousness for built (>=0.5), NDVI for veg (>=0.3)
            mask_built[imp >= 0.5] = 1.0
            mask_veg[ndvi >= 0.3] = 1.0

        # stack channels: order [DSM, impervious, canopy, NDVI, slope]
        x_in = np.stack([dsm, imp, canopy, ndvi, slope], axis=0)
        y_height = height[np.newaxis, :, :]
        y_mask = np.stack([mask_built, mask_veg], axis=0)  # 2 channels

        # normalization: per-channel simple scale (could be improved)
        x_in[0] = x_in[0] / (np.nanmax(x_in[0]) + 1e-6)  # DSM approx scale
        x_in[1:] = (x_in[1:] - np.nanmean(x_in[1:])) / (np.nanstd(x_in[1:]) + 1e-6)

        x_tensor = torch.from_numpy(x_in).float()
        yh = torch.from_numpy(y_height).float()
        ym = torch.from_numpy(y_mask).float()

        if self.transforms:
            x_tensor = self.transforms(x_tensor)

        return x_tensor, yh, ym
